# TrustOCT Interactive Jupyter Notebook
### Title: TrustOCT: A Trustworthy Cross-Device Retinal OCT Disease Classification Framework Using Domain Generalization and Uncertainty-Aware Selective Prediction

This notebook allows running the TrustOCT framework cell-by-cell for interactive training, validation, explainability visualization, and calibration diagnostics.

## Step 1: Environment & Dataset Initialization

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone Repository and navigate to project directory
!git clone https://github.com/Gnanapravallika/Trustworthy-OCT-AI.git
%cd Trustworthy-OCT-AI
!git pull origin main

In [ ]:
# 3. Install Requirements
!pip install -r requirements.txt

In [ ]:
# 4. Authenticate Kaggle and Download Datasets
from google.colab import files
files.upload() # Upload kaggle.json here
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download Kermany & NEH-UT
!kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p datasets/Kermany
!kaggle datasets download -d orvile/neh-ut-oct-dataset --unzip -p datasets/NEH_UT

# Copy OCTID from your Google Drive
!mkdir -p datasets/OCTID
!cp -r "/content/drive/MyDrive/OCTID/." datasets/OCTID/

In [ ]:
# 5. Verify Dataset and generate reports
from src.datasets import verify_dataset, generate_statistics_report

verify_dataset('configs/dataset.yaml', 'outputs/reports/dataset_report.json')
generate_statistics_report('configs/dataset.yaml', 'outputs/reports/dataset_statistics.json', max_samples=1000)

## Step 2: Interactive Model Training & Optimization

In [ ]:
# 1. Import ML packages and custom modules
import os
import random
import numpy as np
import torch
import yaml
from datetime import datetime
import json

from src.models import build_model
from src.datasets import get_dataset_and_loader
from src.trainer import Trainer

In [ ]:
# 2. Set Random Seeds for Reproducibility
def enforce_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seeds enforced to: {seed}")

enforce_seeds(42)

In [ ]:
# 3. Run EXP001 (Baseline ResNet50) Training Session
exp_name = "EXP001_Baseline"
model_cfg_path = "configs/EXP001_Baseline.yaml"
dataset_cfg_path = "configs/dataset.yaml"
train_cfg_path = "configs/train.yaml"
aug_cfg_path = "configs/augmentation.yaml"

# Load loaders
train_dataset, train_loader = get_dataset_and_loader("train", dataset_cfg_path, aug_cfg_path)
val_dataset, val_loader = get_dataset_and_loader("val", dataset_cfg_path, aug_cfg_path)

# Build Model
model = build_model(model_cfg_path)

# Setup output directory
experiment_dir = os.path.join("outputs", exp_name)
os.makedirs(experiment_dir, exist_ok=True)

# Compile Trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    train_config_path=train_cfg_path,
    model_config_path=model_cfg_path,
    experiment_dir=experiment_dir
)

# Start training
trainer.fit()

In [ ]:
# 4. Run EXP006 (Proposed TrustOCT Framework) Training Session
exp_name = "EXP006_TrustOCT"
model_cfg_path = "configs/EXP006_TrustOCT.yaml"
dataset_cfg_path = "configs/dataset.yaml"
train_cfg_path = "configs/train.yaml"
aug_cfg_path = "configs/augmentation.yaml"

enforce_seeds(42)

# Load loaders
train_dataset, train_loader = get_dataset_and_loader("train", dataset_cfg_path, aug_cfg_path)
val_dataset, val_loader = get_dataset_and_loader("val", dataset_cfg_path, aug_cfg_path)

# Build Model
model = build_model(model_cfg_path)

# Setup output directory
experiment_dir = os.path.join("outputs", exp_name)
os.makedirs(experiment_dir, exist_ok=True)

# Compile Trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    train_config_path=train_cfg_path,
    model_config_path=model_cfg_path,
    experiment_dir=experiment_dir
)

# Start training
trainer.fit()

## Step 3: Evaluation, Diagnostics, and Explainability

In [ ]:
# 1. Run cross-dataset external validation on trained models
from src.evaluation import evaluate_cross_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Evaluate Proposed TrustOCT Model
checkpoint = torch.load('outputs/EXP006_TrustOCT/weights_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state'])
model = model.to(device)

results = evaluate_cross_dataset(model, val_loader, device, head_type='edl', num_classes=4)
print("Proposed TrustOCT Validation Metrics:")
for k, v in results.items():
    if k != 'confusion_matrix':
        print(f"  • {k}: {v:.4f}")

In [ ]:
# 2. Generate and save paper-ready plots (Confusion Matrix and Calibration reliability curve)
from src.plots import plot_confusion_matrix, plot_reliability_diagram

plot_confusion_matrix(
    cm=results['confusion_matrix'],
    classes=["CNV", "DME", "DRUSEN", "NORMAL"],
    save_path='outputs/EXP006_TrustOCT/confusion_matrix.png'
)

In [ ]:
# 3. Run visual explanation comparison maps (Grad-CAM vs LayerCAM overlays)
from src.explainability import compare_and_save_visualizations

target_layers_gradcam = [model.backbone.layer4]
target_layers_layercam = [model.backbone.layer3]

# Attributions on a sample NORMAL B-scan image
sample_image = 'datasets/Kermany/OCT2017 /test/NORMAL/NORMAL-1017326-1.jpeg'
if os.path.exists(sample_image):
    compare_and_save_visualizations(
        model=model,
        target_layers_gradcam=target_layers_gradcam,
        target_layers_layercam=target_layers_layercam,
        image_path=sample_image,
        target_class=3,
        output_dir='outputs/figures/explainability',
        prefix='sample_normal'
    )
else:
    print("Verify if sample image path exists on disk.")

In [ ]:
# 4. Zip and download all generated reports, weights, and plots
!zip -r outputs.zip outputs/